In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")

transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=128, shuffle=False)

classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

100%|██████████| 170M/170M [15:07<00:00, 188kB/s]   


In [8]:
class CIFAR10_AdvancedCNN(nn.Module):
    def __init__(self):
        super(CIFAR10_AdvancedCNN, self).__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        self.block3 = nn.Sequential(
            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.Conv2d(256, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Sequential(
            nn.Linear(256 * 4 * 4, 1024),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Linear(512, 10)
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

In [9]:
model = CIFAR10_AdvancedCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=5e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=5, factor=0.5)

In [13]:
loss_values, accuracy_values = [], []

for epoch in range(100):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    epoch_acc = 100 * correct / total
    loss_values.append(running_loss / len(train_loader))
    accuracy_values.append(epoch_acc)
    
    # Cập nhật Scheduler
    scheduler.step(epoch_acc)
    
    print(f"Epoch {epoch+1}/100 | Loss: {loss_values[-1]:.4f} | Acc: {epoch_acc:.2f}% ")

Epoch 1/100 | Loss: 0.8580 | Acc: 69.80% 
Epoch 2/100 | Loss: 0.7813 | Acc: 72.51% 
Epoch 3/100 | Loss: 0.7222 | Acc: 74.70% 
Epoch 4/100 | Loss: 0.6786 | Acc: 76.51% 
Epoch 5/100 | Loss: 0.6335 | Acc: 78.00% 
Epoch 6/100 | Loss: 0.5994 | Acc: 79.25% 
Epoch 7/100 | Loss: 0.5701 | Acc: 80.33% 
Epoch 8/100 | Loss: 0.5453 | Acc: 81.12% 
Epoch 9/100 | Loss: 0.5186 | Acc: 82.01% 
Epoch 10/100 | Loss: 0.4996 | Acc: 82.82% 
Epoch 11/100 | Loss: 0.4835 | Acc: 83.25% 
Epoch 12/100 | Loss: 0.4632 | Acc: 83.93% 
Epoch 13/100 | Loss: 0.4482 | Acc: 84.57% 
Epoch 14/100 | Loss: 0.4375 | Acc: 84.90% 
Epoch 15/100 | Loss: 0.4201 | Acc: 85.42% 
Epoch 16/100 | Loss: 0.4103 | Acc: 85.76% 
Epoch 17/100 | Loss: 0.3949 | Acc: 86.25% 
Epoch 18/100 | Loss: 0.3834 | Acc: 86.66% 
Epoch 19/100 | Loss: 0.3714 | Acc: 87.18% 
Epoch 20/100 | Loss: 0.3647 | Acc: 87.38% 
Epoch 21/100 | Loss: 0.3548 | Acc: 87.80% 
Epoch 22/100 | Loss: 0.3469 | Acc: 87.91% 
Epoch 23/100 | Loss: 0.3433 | Acc: 87.97% 
Epoch 24/100 | Loss:

In [ ]:
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(loss_values, color='b', label='Loss')
plt.grid(True); plt.legend()
plt.subplot(1, 2, 2)
plt.plot(accuracy_values, color='g', label='Accuracy')
plt.axhline(y=90, color='r', linestyle='--') 
plt.grid(True); plt.legend()
plt.show()

In [ ]:
model.eval()
correct, total = 0, 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"\n Độ chính xác trên tập test: {100 * correct / total:.2f}%")

In [ ]:

def visualize_prediction():
    model.eval()
    images, labels = next(iter(test_loader))
    images, labels = images.to(device), labels.to(device)
    
    outputs = model(images)
    _, predicted = torch.max(outputs, 1)
    
    mean, std = np.array([0.4914, 0.4822, 0.4465]), np.array([0.2023, 0.1994, 0.2010])
    
    fig, axes = plt.subplots(1, 5, figsize=(15, 4))
    for i in range(5):
        img = images[i].cpu().permute(1, 2, 0).numpy() * std + mean
        axes[i].imshow(np.clip(img, 0, 1))
        
        color = 'green' if predicted[i] == labels[i] else 'red'
        axes[i].set_title(f"Dự đoán: {classes[predicted[i]]}\nThật: {classes[labels[i]]}", color=color, fontsize=9)
        axes[i].axis('off')
    
    plt.tight_layout()
    plt.show()

visualize_prediction()